# Binary fracture detector — 3-dataset fine-tuning

This is a **one-class** detector: every verified box is `fracture`. It deliberately does not predict anatomy, which prevents the old false shoulder labelling problem.

## Kaggle setup

Enable a GPU and add exactly these four inputs:

1. `BoneFractureYolo8` dataset
2. `FracAtlas` dataset
3. `graz-yolo-ready-v11` (preferred) **or** the raw `GRAZPEDWRI-DX` input, provided `folder_structure` contains YOLO `.txt` labels
4. A **new private Kaggle dataset containing the currently deployed** `best_binary_fracture.pt` from this repository. Do not use the 30%-mAP fine-tuned checkpoint as the starting model.

The notebook checks all inputs, builds a clean binary dataset, measures the baseline, trains for at most nine GPU hours, evaluates again, and exports the better checkpoint as `/kaggle/working/best_binary_fracture.pt`.

In [ ]:
%pip install -q "ultralytics>=8.2.0"

import csv
import json
import random
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import torch
import yaml
from PIL import Image, ImageFile
from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = True
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. Kaggle: Settings → Accelerator → GPU, restart, then Run All.')

SEED = 42
DEVICE = 0
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
DATASET = WORK / 'binary_fracture_3data'
random.seed(SEED)
print('GPU:', torch.cuda.get_device_name(DEVICE))

def find_dir(name, required):
    matches = [p for p in INPUT.rglob(name) if p.is_dir() and all((p / item).exists() for item in required)]
    return min(matches, key=lambda p: len(p.parts)) if matches else None

BONE8 = find_dir('BoneFractureYolo8', ('train', 'valid', 'test'))
FRACATLAS = find_dir('FracAtlas', ('images', 'Annotations'))
GRAZ = None
GRAZ_MODE = None
for candidate in INPUT.rglob('data.yaml'):
    root = candidate.parent
    if (root / 'images').is_dir() and (root / 'labels').is_dir() and 'graz' in str(root).lower():
        GRAZ, GRAZ_MODE = root, 'ready'
        break

# Kaggle occasionally fails to mount the ready-made dataset. The raw source is also supported
# when its folder_structure contains the published YOLO annotation files.
if GRAZ is None:
    raw_matches = [p for p in INPUT.rglob('GRAZPEDWRI-DX') if p.is_dir() and (p / 'folder_structure').is_dir() and any(p.glob('images_part*'))]
    if raw_matches:
        GRAZ, GRAZ_MODE = min(raw_matches, key=lambda p: len(p.parts)), 'raw'

for label, path in [('BoneFractureYolo8', BONE8), ('FracAtlas', FRACATLAS), ('graz-yolo-ready-v11', GRAZ)]:
    print(f'{label}: {path}')
    if path is None:
        raise FileNotFoundError(f'{label} was not found. Add the required Kaggle input, then restart.')
print('GRAZ mode:', GRAZ_MODE)
print('All three training datasets were found.')


In [ ]:
# Build a high-quality one-class dataset. Unknown/unlabelled images are never silently treated as normal.
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ('train', 'val', 'test'):
    (DATASET / split / 'images').mkdir(parents=True)
    (DATASET / split / 'labels').mkdir(parents=True)

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp'}
stats = Counter()
serial = Counter()

def image_files(folder):
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def readable(image):
    try:
        with Image.open(image) as opened:
            opened.verify()
        return True
    except Exception:
        stats['unreadable_images'] += 1
        return False

def as_box(values):
    if len(values) == 4:
        x, y, w, h = values
    elif len(values) >= 6 and len(values) % 2 == 0:  # segmentation polygon → bounding box
        xs, ys = values[0::2], values[1::2]
        xmin, xmax = max(0.0, min(xs)), min(1.0, max(xs))
        ymin, ymax = max(0.0, min(ys)), min(1.0, max(ys))
        x, y, w, h = (xmin + xmax) / 2, (ymin + ymax) / 2, xmax - xmin, ymax - ymin
    else:
        return None
    if not (0 < w <= 1 and 0 < h <= 1 and 0 <= x <= 1 and 0 <= y <= 1):
        return None
    return x, y, w, h

def yolo_boxes(label_path, allowed_classes=None):
    boxes, seen_classes = [], set()
    if not label_path.is_file():
        return boxes, seen_classes
    for row in label_path.read_text(errors='ignore').splitlines():
        parts = row.split()
        try:
            source_class = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except (IndexError, ValueError):
            continue
        seen_classes.add(source_class)
        if allowed_classes is not None and source_class not in allowed_classes:
            continue
        box = as_box(values)
        if box is not None:
            boxes.append('0 ' + ' '.join(f'{value:.6f}' for value in box))
    return boxes, seen_classes

def add_sample(image, boxes, split, source, positive):
    if not readable(image):
        return
    serial[(split, source)] += 1
    name = f'{source}_{serial[(split, source)]:06d}_{image.stem}'
    shutil.copy2(image, DATASET / split / 'images' / f'{name}{image.suffix.lower()}')
    (DATASET / split / 'labels' / f'{name}.txt').write_text('\n'.join(boxes))
    stats[f'{source}_{split}_{"positive" if positive else "negative"}'] += 1
    stats[f'{source}_boxes'] += len(boxes)

negative_candidates = []

# 1) BoneFractureYolo8: only confirmed fracture classes. Class 4 is an ambiguous generic humerus label.
BONE_FRACTURE_CLASSES = {0, 1, 2, 3, 5, 6}
AMBIGUOUS_BONE_CLASS = 4
for source_split, split in {'train': 'train', 'valid': 'val', 'test': 'test'}.items():
    image_dir = BONE8 / source_split / 'images'
    label_dir = BONE8 / source_split / 'labels'
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f'Broken BoneFractureYolo8 split: {source_split}')
    for image in image_files(image_dir):
        boxes, all_classes = yolo_boxes(label_dir / f'{image.stem}.txt', BONE_FRACTURE_CLASSES)
        if AMBIGUOUS_BONE_CLASS in all_classes:
            stats['bone_ambiguous_skipped'] += 1
        elif boxes:
            add_sample(image, boxes, split, 'bone', True)

# 2) FracAtlas: its Fractured/Non_fractured folders give verified binary status.
frac_labels = next((p for p in [FRACATLAS / 'Annotations' / 'YOLO', FRACATLAS / 'Annotations' / 'yolo'] if p.is_dir()), None)
if frac_labels is None:
    frac_labels = next((p for p in FRACATLAS.rglob('*') if p.is_dir() and p.name.lower() == 'yolo' and any(p.glob('*.txt'))), None)
if frac_labels is None:
    raise FileNotFoundError('FracAtlas YOLO annotations were not found.')
fractured_dir = FRACATLAS / 'images' / 'Fractured'
normal_dir = FRACATLAS / 'images' / 'Non_fractured'
if not normal_dir.exists():
    normal_dir = FRACATLAS / 'images' / 'Non-Fractured'
if not fractured_dir.is_dir() or not normal_dir.is_dir():
    raise FileNotFoundError('FracAtlas Fractured and Non_fractured folders are required.')
frac_positive = []
for image in image_files(fractured_dir):
    boxes, _ = yolo_boxes(frac_labels / f'{image.stem}.txt')
    if boxes:
        frac_positive.append((image, boxes))
    else:
        stats['frac_positive_without_box_skipped'] += 1
random.shuffle(frac_positive)
n_train, n_val = int(.75 * len(frac_positive)), int(.90 * len(frac_positive))
for split, samples in [('train', frac_positive[:n_train]), ('val', frac_positive[n_train:n_val]), ('test', frac_positive[n_val:])]:
    for image, boxes in samples:
        add_sample(image, boxes, split, 'fracatlas', True)
normal_images = image_files(normal_dir)
random.shuffle(normal_images)
for index, image in enumerate(normal_images):
    split = 'train' if index < .75 * len(normal_images) else ('val' if index < .90 * len(normal_images) else 'test')
    negative_candidates.append((image, split, 'fracatlas'))

# 3) GRAZPEDWRI-DX: retain only its verified fracture class. Other labelled findings may be negatives.
if GRAZ_MODE == 'ready':
    graz_config = yaml.safe_load((GRAZ / 'data.yaml').read_text())
    graz_names = graz_config.get('names', {})
    if isinstance(graz_names, list):
        graz_names = dict(enumerate(graz_names))
    graz_fracture_ids = {int(key) for key, value in graz_names.items() if 'fracture' in str(value).lower()}
    if len(graz_fracture_ids) != 1:
        raise RuntimeError(f'Could not identify exactly one GRAZ fracture class: {graz_names}')
    print('GRAZ fracture class:', graz_fracture_ids)
    for source_split in ('train', 'valid', 'val', 'test'):
        image_dir = GRAZ / 'images' / source_split
        label_dir = GRAZ / 'labels' / source_split
        if not image_dir.is_dir():
            image_dir, label_dir = GRAZ / source_split / 'images', GRAZ / source_split / 'labels'
        if not image_dir.is_dir():
            continue
        split = 'val' if source_split in ('valid', 'val') else source_split
        for image in image_files(image_dir):
            label_path = label_dir / f'{image.stem}.txt'
            boxes, seen_classes = yolo_boxes(label_path, graz_fracture_ids)
            if boxes:
                add_sample(image, boxes, split, 'graz', True)
            elif label_path.is_file() and seen_classes:
                negative_candidates.append((image, split, 'graz'))
else:
    # Raw Figshare layout. Its published YOLO export uses class 3 for fracture.
    candidate_dirs = [GRAZ / 'folder_structure' / name for name in ('labels', 'yolov5', 'annotations')]
    candidate_dirs += [p for p in (GRAZ / 'folder_structure').rglob('*') if p.is_dir() and any(p.glob('*.txt'))]
    label_dir = next((p for p in candidate_dirs if p.is_dir() and any(p.rglob('*.txt'))), None)
    if label_dir is None:
        raise FileNotFoundError('Raw GRAZPEDWRI-DX has no YOLO .txt labels under folder_structure. It cannot be used for box training.')
    labels_by_stem = {p.stem: p for p in label_dir.rglob('*.txt')}
    images_by_stem = {}
    for part in sorted(GRAZ.glob('images_part*')):
        for image in image_files(part):
            images_by_stem.setdefault(image.stem, image)
    if not images_by_stem or not labels_by_stem:
        raise RuntimeError('Raw GRAZPEDWRI-DX image or label files are empty.')
    patient_by_stem = {}
    csv_path = GRAZ / 'dataset.csv'
    if csv_path.is_file():
        with csv_path.open(newline='', encoding='utf-8', errors='ignore') as handle:
            for row in csv.DictReader(handle):
                stem = str(row.get('filestem') or row.get('filename') or row.get('image_id') or '').strip()
                patient_by_stem[stem] = str(row.get('patient_id') or row.get('patientid') or stem).strip()
    patient_ids = sorted({patient_by_stem.get(stem, stem) for stem in images_by_stem})
    random.shuffle(patient_ids)
    train_patients = set(patient_ids[:int(.75 * len(patient_ids))])
    val_patients = set(patient_ids[int(.75 * len(patient_ids)):int(.90 * len(patient_ids))])
    print(f'Raw GRAZ: {len(images_by_stem):,} images, {len(labels_by_stem):,} labels; using documented fracture class {{3}}')
    for stem, image in images_by_stem.items():
        patient_id = patient_by_stem.get(stem, stem)
        split = 'train' if patient_id in train_patients else ('val' if patient_id in val_patients else 'test')
        label_path = labels_by_stem.get(stem)
        boxes, seen_classes = yolo_boxes(label_path, {3}) if label_path else ([], set())
        if boxes:
            add_sample(image, boxes, split, 'graz', True)
        elif label_path is not None and seen_classes:
            negative_candidates.append((image, split, 'graz'))

# Cap verified negatives at 2× the positive examples. This protects recall instead of letting normal X-rays dominate.
positive_total = sum(value for key, value in stats.items() if key.endswith('_positive'))
random.shuffle(negative_candidates)
negative_limit = min(len(negative_candidates), 2 * positive_total)
for image, split, source in negative_candidates[:negative_limit]:
    add_sample(image, [], split, source, False)
stats['negative_candidates'] = len(negative_candidates)
stats['negative_used'] = negative_limit

data_yaml = WORK / 'binary_fracture_3data.yaml'
data_yaml.write_text(yaml.safe_dump({'path': str(DATASET), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'nc': 1, 'names': ['fracture']}, sort_keys=False))
print('\nDATASET SUMMARY')
for split in ('train', 'val', 'test'):
    images = len(list((DATASET / split / 'images').glob('*')))
    labels = len(list((DATASET / split / 'labels').glob('*')))
    assert images == labels and images > 0, f'Invalid {split} split'
    print(f'{split}: {images:,} images')
for key in sorted(stats):
    print(f'  {key}: {stats[key]:,}')
assert positive_total > 0 and stats['graz_boxes'] > 0, 'No usable fracture boxes were found.'
print('Dataset YAML:', data_yaml)


In [ ]:
# Locate ONLY the current deployed binary checkpoint. If multiple checkpoint datasets are attached, stop instead of guessing.
checkpoint_files = []
checkpoint_dirs = []
for name in ('best_binary_fracture.pt', 'best.pt', 'best'):
    checkpoint_files.extend(p for p in INPUT.rglob(name) if p.is_file())
    checkpoint_dirs.extend(p for p in INPUT.rglob(name) if p.is_dir() and (p / 'data.pkl').is_file())
checkpoint_files = sorted(set(checkpoint_files), key=lambda p: len(p.parts))
checkpoint_dirs = sorted(set(checkpoint_dirs), key=lambda p: len(p.parts))
print('Checkpoint files:', *checkpoint_files, sep='\n  ')
print('Checkpoint archives:', *checkpoint_dirs, sep='\n  ')
if len(checkpoint_files) + len(checkpoint_dirs) != 1:
    raise RuntimeError('Attach exactly ONE checkpoint input: the current deployed best_binary_fracture.pt. Remove old fine-tuning outputs from Kaggle Inputs.')
if checkpoint_files:
    baseline_path = checkpoint_files[0]
else:
    checkpoint_dir = checkpoint_dirs[0]
    baseline_path = WORK / 'reconstructed_current_binary.pt'
    with zipfile.ZipFile(baseline_path, 'w', compression=zipfile.ZIP_STORED) as archive:
        for file_path in checkpoint_dir.rglob('*'):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(checkpoint_dir.parent).as_posix())

baseline = YOLO(str(baseline_path))
if [baseline.names[index] for index in sorted(baseline.names)] != ['fracture']:
    raise RuntimeError(f'Wrong checkpoint. Expected one binary fracture class, got: {baseline.names}')
print('Baseline checkpoint:', baseline_path)

# A pre-training validation makes the selection evidence-based. The test split stays untouched until the final report.
baseline_val = baseline.val(data=str(data_yaml), split='val', imgsz=800, batch=4, device=DEVICE, workers=2, plots=False, verbose=False)
baseline_metrics = {'precision': float(baseline_val.box.mp), 'recall': float(baseline_val.box.mr), 'map50': float(baseline_val.box.map50), 'map50_95': float(baseline_val.box.map)}
print('BASELINE VALIDATION:', baseline_metrics)

# Fine-tune gently at higher resolution for small hand/wrist fractures. Nine hours fits within a Kaggle GPU session without forcing an OOM-prone batch size.
run_name = 'binary_3data_finetune'
results = baseline.train(
    data=str(data_yaml), epochs=200, time=9.0, imgsz=800, batch=8, device=DEVICE, workers=2,
    project=str(WORK / 'runs'), name=run_name, exist_ok=True,
    optimizer='AdamW', lr0=0.00005, lrf=0.1, cos_lr=True, patience=40, seed=SEED,
    mosaic=0.20, close_mosaic=15, mixup=0.0, copy_paste=0.0,
    degrees=5.0, translate=0.03, scale=0.25, fliplr=0.5, flipud=0.0, perspective=0.0, erasing=0.0,
    cache=False, amp=True, save=True, save_period=10, plots=True,
)

run_dir = Path(getattr(results, 'save_dir', WORK / 'runs' / run_name))
trained_path = run_dir / 'weights' / 'best.pt'
if not trained_path.is_file():
    raise FileNotFoundError(f'Training finished without best.pt: {trained_path}')
trained = YOLO(str(trained_path))
trained_val = trained.val(data=str(data_yaml), split='val', imgsz=800, batch=4, device=DEVICE, workers=2, plots=False, verbose=False)
trained_metrics = {'precision': float(trained_val.box.mp), 'recall': float(trained_val.box.mr), 'map50': float(trained_val.box.map50), 'map50_95': float(trained_val.box.map)}
print('FINE-TUNED VALIDATION:', trained_metrics)

# Keep the fine-tuned model only when it improves quality, or makes a meaningful recall gain without materially harming localization.
quality_win = trained_metrics['map50'] >= baseline_metrics['map50'] + 0.005 and trained_metrics['recall'] >= baseline_metrics['recall'] - 0.02
sensitivity_win = trained_metrics['recall'] >= baseline_metrics['recall'] + 0.03 and trained_metrics['map50'] >= baseline_metrics['map50'] - 0.015
use_finetuned = bool(quality_win or sensitivity_win)
selected_path = trained_path if use_finetuned else baseline_path
published = WORK / 'best_binary_fracture.pt'
shutil.copy2(selected_path, published)
selection = {'selected': 'fine_tuned' if use_finetuned else 'baseline', 'baseline_validation': baseline_metrics, 'fine_tuned_validation': trained_metrics, 'reason': 'quality_win' if quality_win else ('sensitivity_win' if sensitivity_win else 'baseline_protected'), 'published_checkpoint': str(published)}
(WORK / 'model_selection.json').write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))


In [ ]:
# Final, untouched test-set report for the selected checkpoint.
selected = YOLO(str(published))
test = selected.val(data=str(data_yaml), split='test', imgsz=800, batch=4, device=DEVICE, workers=2, plots=True, verbose=False, project=str(WORK / 'final_evaluation'), name='test')
test_metrics = {'precision': float(test.box.mp), 'recall': float(test.box.mr), 'map50': float(test.box.map50), 'map50_95': float(test.box.map)}
selection['test_metrics'] = test_metrics
(WORK / 'model_selection.json').write_text(json.dumps(selection, indent=2))
print('\nFINAL RESULT')
print('Published model:', published)
print('Selected:', selection['selected'], '|', selection['reason'])
print('Held-out test metrics:', test_metrics)
print('\nDownload these Kaggle outputs:')
print('  1.', published)
print('  2.', WORK / 'model_selection.json')
print('  3.', WORK / 'runs' / run_name / 'results.csv')
print('Do not deploy a checkpoint until these three files are available.')
